# Baselines on training set v3

**Why:** every score so far (0.9330, 0.8034 …) was measured on **v2**. v3 has new rules (real words, one text
one split, warm-up cut) and **the 19 SQL features**. DistilBERT needs a fresh bar to beat.

**What:** the same cheap recipe as `training/verify_ablation.py` — TF-IDF + logistic regression — so v2 and v3
are measured the same way. Train on `train`, score on `test` (2024). Ranking metrics only, so the case-control
sampling doesn't matter here.

**Three scores** (CLAUDE.md trap 5): **pooled** (flattered), **within product × issue**, **within company**
(what one bank sees — the honest one).

In [1]:
# 1. load v3 and prove it is the exported file
import hashlib, json
import numpy as np, pandas as pd
from pathlib import Path
DIR = Path("../Data/data/interim")
manifest = json.loads((DIR / "triageiq_training_v3.manifest.json").read_text())
pq = DIR / manifest["file"]
assert hashlib.sha256(pq.read_bytes()).hexdigest() == manifest["sha256"], "file differs from the manifest"
df = pd.read_parquet(pq)
tr = df[df.split == "train"].reset_index(drop=True)
te = df[df.split == "test"].reset_index(drop=True)
print(f"sha256 OK · train {len(tr):,} ({tr.paid.mean():.1%} payouts) · test {len(te):,} ({te.paid.mean():.2%} payouts)")

sha256 OK · train 62,940 (25.0% payouts) · test 150,000 (2.31% payouts)


In [2]:
# 2. the three scores — same rules as verify_ablation.py
from sklearn.metrics import roc_auc_score, average_precision_score
def within(te, y, p, keys, min_pos, min_n):
    aucs, wts = [], []
    for _, idx in te.groupby(keys, observed=True).indices.items():
        yy = y[idx]
        if yy.sum() < min_pos or yy.sum() == len(yy) or len(yy) < min_n:
            continue
        aucs.append(roc_auc_score(yy, p[idx])); wts.append(len(yy))
    return float(np.average(aucs, weights=np.array(wts, float))), len(aucs)
def score(p, y=te.paid.values):
    ws, ns = within(te, y, p, ["product_id", "issue_id"], 25, 150)   # within product x issue
    wc, nc = within(te, y, p, ["company_id"], 20, 200)               # within one company
    return {"pooled_auc": roc_auc_score(y, p), "pooled_pr": average_precision_score(y, p),
            "within_strata_auc": ws, "n_strata": ns, "within_company_auc": wc, "n_companies": nc}

In [3]:
# 3. inputs — every transform is FIT ON TRAIN ONLY, then applied to test (guardrail §10)
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
cols = manifest["columns"]
CAT, NUM = cols["categorical"], cols["flags"] + cols["numeric"]
WEAK = ["company_trend_90d", "issue_trend_90d", "company_quiet_days"]     # weak alone (AUC 0.50-0.58)

vec = TfidfVectorizer(max_features=80_000, ngram_range=(1, 2), min_df=5, sublinear_tf=True, strip_accents="unicode")
Xtr_t, Xte_t = vec.fit_transform(tr.text), vec.transform(te.text)

def tab(names):
    oh, sc = OneHotEncoder(handle_unknown="ignore"), StandardScaler()
    A = sparse.hstack([oh.fit_transform(tr[CAT]), sparse.csr_matrix(sc.fit_transform(tr[names]))]).tocsr()
    B = sparse.hstack([oh.transform(te[CAT]),     sparse.csr_matrix(sc.transform(te[names]))]).tocsr()
    return A, B
Xtr_f, Xte_f = tab(NUM)                                    # all 19 inputs
Xtr_w, Xte_w = tab([c for c in NUM if c not in WEAK])      # without the 3 weak ones
print("text", Xtr_t.shape, "· features", Xtr_f.shape, "· features minus weak", Xtr_w.shape)

text (62940, 80000) · features (62940, 228) · features minus weak (62940, 225)


In [4]:
# 4. fit and score — text only · features only · both
from sklearn.linear_model import LogisticRegression
runs = {
    "text":                     (Xtr_t, Xte_t),
    "features (19)":            (Xtr_f, Xte_f),
    "features minus weak (16)": (Xtr_w, Xte_w),
    "fusion":                   (sparse.hstack([Xtr_t, Xtr_f]).tocsr(), sparse.hstack([Xte_t, Xte_f]).tocsr()),
    "fusion minus weak":        (sparse.hstack([Xtr_t, Xtr_w]).tocsr(), sparse.hstack([Xte_t, Xte_w]).tocsr()),
}
res, models, preds = {}, {}, {}
for name, (A, B) in runs.items():
    m = LogisticRegression(max_iter=800, solver="liblinear").fit(A, tr.paid.values)
    preds[name] = m.predict_proba(B)[:, 1]
    res[name], models[name] = score(preds[name]), m
table = pd.DataFrame(res).T
print(table[["pooled_auc", "pooled_pr", "within_strata_auc", "within_company_auc"]].round(4).to_string())
print(f"\nstrata scored: {int(table.n_strata.iloc[0])} · companies scored: {int(table.n_companies.iloc[0])}")

                          pooled_auc  pooled_pr  within_strata_auc  within_company_auc
text                          0.9599     0.3523             0.8916              0.7956
features (19)                 0.9642     0.3645             0.9276              0.7500
features minus weak (16)      0.9641     0.3631             0.9272              0.7492
fusion                        0.9713     0.4262             0.9457              0.8115
fusion minus weak             0.9713     0.4254             0.9454              0.8106

strata scored: 20 · companies scored: 24


In [5]:
# 5. against v2 (training/ablation_results.json, section B) — NOT the same test set, see the note below
v2 = json.loads(Path("ablation_results.json").read_text())["B_shipped_v2_artifact"]
rows = []
for v3_name, v2_name in [("text", "text"), ("features (19)", "metadata"), ("fusion", "fusion")]:
    for metric in ["pooled_auc", "within_strata_auc", "within_company_auc"]:
        rows.append({"model": v3_name, "metric": metric, "v2": round(v2[v2_name][metric], 4), "v3": round(res[v3_name][metric], 4)})
print(pd.DataFrame(rows).pivot(index="model", columns="metric", values=["v2", "v3"]).round(4).to_string())

                      v2                                              v3                                     
metric        pooled_auc within_company_auc within_strata_auc pooled_auc within_company_auc within_strata_auc
model                                                                                                        
features (19)     0.9533             0.7463            0.9076     0.9642             0.7500            0.9276
fusion            0.9634             0.8034            0.9330     0.9713             0.8115            0.9457
text              0.9542             0.7900            0.8905     0.9599             0.7956            0.8916


In [6]:
# 6. which of the 15 number-inputs does the features model lean on? (standardized coefficients)
m = models["features (19)"]
coef = pd.Series(m.coef_[0][-len(NUM):], index=NUM).sort_values(key=abs, ascending=False)
print(coef.round(3).to_string())

company_issue_share_90d    -2.230
company_rate_s              0.811
product_rate_s             -0.625
company_issue_rate_s        0.436
company_share_90d          -0.369
company_untimely_rate      -0.196
issue_rate_s               -0.144
issue_trend_90d             0.093
issue_share_90d             0.092
company_trend_90d          -0.077
company_issue_no_history   -0.062
company_quiet_days          0.049
company_no_history          0.048
is_older_american           0.044
is_servicemember           -0.007


In [7]:
# 6b. the 512 question: are long complaints HARDER, or does cutting them at 512 LOSE information?
#     TF-IDF reads the WHOLE text. Split the test set exactly as fusion_distilbert.ipynb does
#     (DistilBERT tokens, markers as single tokens, > 510 = cut) and score each half.
#     TF-IDF also worse on 'cut'  -> long complaints are simply harder.
#     TF-IDF fine, DistilBERT worse -> the cut loses information -> try head + tail truncation.
import os; os.environ.setdefault("USE_TF", "0")
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tok.add_special_tokens({"additional_special_tokens": manifest["text_markers"]})
n_tok = np.array([len(x) for x in tok(list(te.text), add_special_tokens=False, truncation=False)["input_ids"]])
cut = n_tok > 510
y = te.paid.values
rows = {}
for name in ["text", "features (19)", "fusion"]:
    rows[name] = {"auc_fits_512": roc_auc_score(y[~cut], preds[name][~cut]),
                  "auc_cut_at_512": roc_auc_score(y[cut], preds[name][cut])}
slice512 = pd.DataFrame(rows).T
print(f"test complaints cut at 512: {cut.sum():,} ({cut.mean():.1%}) · payouts among them {y[cut].sum()} "
      f"({y[cut].mean():.2%}) vs {y[~cut].mean():.2%} in the rest")
print(slice512.round(4).to_string())

/opt/anaconda3/envs/ml_ds_env/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.6.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (624 > 512). Running this sequence through the model will result in indexing errors
test complaints cut at 512: 12,602 (8.4%) · payouts among them 430 (3.41%) vs 2.21% in the rest
               auc_fits_512  auc_cut_at_512
text                 0.9603          0.9542
features (19)        0.9656          0.9495
fusion               0.9722          0.9614


In [8]:
# 6c. the business number: if seniors read only the riskiest X% of complaints, what share of all payouts
#     do they catch? (pooled across companies — the whole 2024 test set)
y = te.paid.values
recall_at = {}
for name in ["text", "features (19)", "fusion"]:
    order = np.argsort(-preds[name])
    recall_at[name] = {f"top_{int(s*100)}pct": float(y[order[:int(len(y) * s)]].sum() / y.sum()) for s in (0.05, 0.10, 0.20)}
print(pd.DataFrame(recall_at).T.round(3).to_string())

               top_5pct  top_10pct  top_20pct
text              0.639      0.872      0.980
features (19)     0.651      0.888      0.986
fusion            0.723      0.917      0.993


In [9]:
# 7. save the numbers — code for every quoted number (ground rule 6)
out = {"data": {"file": manifest["file"], "sha256": manifest["sha256"], "n_train": len(tr), "n_test": len(te),
                "test_base_rate": float(te.paid.mean())},
       "results": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in res.items()},
       "slice_512": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in rows.items()},
       "recall_at": recall_at}
Path("baseline_v3_results.json").write_text(json.dumps(out, indent=2))
print("wrote training/baseline_v3_results.json")

wrote training/baseline_v3_results.json
